# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sujan-lab-cell/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Overview of the Action Queue

The goal of this section is to transform the validated **Week 6 client-grouped Random Forest prediction methodology** into a page-level, decision-support **action queue**.

Rather than treating model outputs as automated decision-makers, this queue serves as a **prioritization system** to help human content and SEO teams decide which pages deserve detailed review first under constrained resource budgets.

### Methodology & Out-of-Fold (OOF) Model Scores

In Week 6, we established that a 5-fold `GroupKFold` split by client provides an honest, leakage-free evaluation of cross-client generalization, achieving a conservative **Precision@50 of 0.4440** (beating the rule baseline of 0.3920).

Because individual page-level out-of-fold scores were not saved in Week 6, we reproduce the exact same validated procedure to generate an out-of-fold prediction score (`model_score`) for every eligible page:

1. **Eligible Population:** Filtered to 16,513 eligible pages across 36 clients matching canonical rules (`impressions_total >= 1000` and `april_clicks >= 10`).
2. **Feature Matrix:** 9 pre-May historical features (`impressions_total`, `clicks_total`, `april_impressions`, `april_clicks`, `feb_clicks`, `momentum`, `ctr`, `active_days`, `weighted_position`).
3. **Decline Target:** Future outcome `decline = (may_clicks < 0.8 × april_clicks)` (observed decline base rate of 41.62%).
4. **GroupKFold (5 Folds):** Grouped strictly by `client_hash_id` with 0 client overlap across all training and validation folds.
5. **Score Generation:** For each fold, a `RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)` is trained on training clients and predicts positive-class probabilities on unseen validation clients.

Every eligible page receives exactly one OOF prediction score (`model_score`), representing the model's estimated probability of future click decline under unseen-client validation.

### Deterministic Ranking Rules

The action queue is sorted deterministically using pre-May signals and model outputs only:

1. `model_score` **descending** (highest estimated decline risk first)
2. `april_clicks` **descending** (prioritize higher-traffic pages among equal model scores)
3. `content_hash_id` **ascending** (deterministic tie-breaking)

> [!IMPORTANT]
> **No Target Leakage:** May clicks, May impressions, decline labels, or trend metrics are **never** used as ranking inputs or features.

### Explicit Reason Code Hierarchy & Supporting Signals

To make recommendations transparent and explainable for human reviewers, each page receives the **first applicable reason code** based on the following explicit decision hierarchy:

1. `decline_and_stale`:
   - **Condition:** `model_score >= 0.50` **AND** `days_since_last_update >= 91` days
   - *Explanation:* High decline risk score combined with verified content staleness (>=91 days since last update). Week 4 descriptive findings showed that pages stale for 91+ days had an observed decline rate of 60.85% (+9.65 percentage points higher than non-stale pages). Staleness is defined strictly by update age, never by momentum.

2. `decline_and_low_visibility`:
   - **Condition:** `model_score >= 0.50` **AND** `weighted_position > 15.0`
   - *Explanation:* High decline risk score combined with weak search visibility (average Google ranking position beyond page 1/2).

3. `decline_recent_update`:
   - **Condition:** `model_score >= 0.50` **AND** `days_since_last_update < 91` days
   - *Explanation:* High decline risk score on a page that was updated within the last 90 days. Indicates potential need for pre-refresh investigation to avoid content cannibalization.

4. `model_signal_only`:
   - **Condition:** `model_score >= 0.50` (neither stale nor low-visibility trigger met)
   - *Explanation:* High decline risk score driven by combined multi-feature historical pattern, without a single secondary staleness or position trigger.

5. `monitor`:
   - **Condition:** `model_score < 0.50`
   - *Explanation:* Lower model score / weaker prioritization signal.

### Suggested Actions & Review Priority

Each reason code maps directly to a suggested human review action and review priority:

| Reason Code | Condition Rule | Suggested Action | Review Priority | Recommended Human Review Focus |
| :--- | :--- | :--- | :---: | :--- |
| `decline_and_stale` | `score >= 0.50` & `days >= 91` | `refresh_review` | **High** | Review out-of-date sections, factual freshness, and temporal content relevance |
| `decline_and_low_visibility` | `score >= 0.50` & `pos > 15.0` | `seo_content_review` | **High** | Audit on-page SEO, title tags, internal linking, and query-intent alignment |
| `decline_recent_update` | `score >= 0.50` & `days < 91` | `investigate_before_refresh` | **High** | Inspect recent changes before re-editing to avoid content cannibalization |
| `model_signal_only` | `score >= 0.50` (other) | `manual_investigation` | **Medium** | Conduct general manual inspection of page performance and query trends |
| `monitor` | `score < 0.50` | `monitor` | **Monitor** | No immediate action required; monitor in regular analytics cycles |

### Guardrails & Decision-Support Framing

- **Decision Support, Not Automation:** The queue produces recommendations for human review, **not** automated production edits or automatic content updates.
- **Language Policy:** Findings describe *observed* patterns, *measured* comparisons, and *ranked* priority scores. They do **not** claim causality or algorithm prediction.

In [1]:
import os
import gc
import numpy as np
import pandas as pd
import polars as pl
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier

print('==================================================')
print('SECTION 1: RANKED ACTIONS + REASON CODES')
print('==================================================')

# 1. Obtain HF_TOKEN to access gated FlyRank/internship-warehouse dataset
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    env_file = Path('.env')
    if not env_file.exists():
        env_file = Path('../.env')
    if env_file.exists():
        for line in env_file.read_text().splitlines():
            if line.startswith('HF_TOKEN='):
                HF_TOKEN = line.split('=', 1)[1].strip().strip('"\'')

if not HF_TOKEN:
    print('[WARNING] HF_TOKEN not found in environment or .env file.')
    print('Please provide your Hugging Face READ token to stream real warehouse daily partitions.')
    raise ValueError('HF_TOKEN required to load real gated warehouse dataset FlyRank/internship-warehouse.')

# 2. Download Feb-May 2026 daily performance partitions from Hugging Face
from huggingface_hub import snapshot_download
local_dir = snapshot_download(
    repo_id='FlyRank/internship-warehouse',
    repo_type='dataset',
    allow_patterns=[
        'dim_*.parquet',
        'fact_content_daily_performance/month=2026-02/*.parquet',
        'fact_content_daily_performance/month=2026-03/*.parquet',
        'fact_content_daily_performance/month=2026-04/*.parquet',
        'fact_content_daily_performance/month=2026-05/*.parquet'
    ],
    token=HF_TOKEN
)

fact_files = list(Path(local_dir).glob('fact_content_daily_performance/**/*.parquet'))
print(f'Found {len(fact_files)} daily parquet partition files.')

# 3. Streamed Lazy Aggregation for canonical Week 5/6 population & features
lazy_daily = pl.scan_parquet(fact_files).select([
    pl.col('report_date').cast(pl.Utf8),
    pl.col('client_hash_id').cast(pl.Categorical),
    pl.col('content_hash_id').cast(pl.Categorical),
    pl.col('gsc_clicks').cast(pl.Int32),
    pl.col('gsc_impressions').cast(pl.Int32),
    pl.col('gsc_avg_position').cast(pl.Float32)
])

# Memory-efficient lazy duplicate check at daily grain
dup_check = lazy_daily.group_by(['report_date', 'client_hash_id', 'content_hash_id']).len().filter(pl.col('len') > 1).select(pl.len()).collect()
dup_count = dup_check[0, 0] if len(dup_check) > 0 else 0
print(f'1. Grain check (report_date x client_hash_id x content_hash_id): duplicates = {dup_count}')
assert dup_count == 0, 'Duplicate rows detected at daily grain!'

feb_mask = (pl.col('report_date') >= '2026-02-01') & (pl.col('report_date') <= '2026-02-28')
mar_mask = (pl.col('report_date') >= '2026-03-01') & (pl.col('report_date') <= '2026-03-31')
apr_mask = (pl.col('report_date') >= '2026-04-01') & (pl.col('report_date') <= '2026-04-30')
pre_may_mask = (pl.col('report_date') >= '2026-02-01') & (pl.col('report_date') <= '2026-04-30')
may_mask = (pl.col('report_date') >= '2026-05-01') & (pl.col('report_date') <= '2026-05-31')

lazy_main_agg = lazy_daily.group_by(['client_hash_id', 'content_hash_id']).agg([
    pl.col('gsc_clicks').filter(feb_mask).sum().alias('feb_clicks'),
    pl.col('gsc_impressions').filter(feb_mask).sum().alias('feb_impressions'),
    pl.col('gsc_clicks').filter(mar_mask).sum().alias('march_clicks'),
    pl.col('gsc_impressions').filter(mar_mask).sum().alias('march_impressions'),
    pl.col('gsc_clicks').filter(apr_mask).sum().alias('april_clicks'),
    pl.col('gsc_impressions').filter(apr_mask).sum().alias('april_impressions'),
    pl.col('gsc_impressions').filter(pre_may_mask).sum().alias('impressions_total'),
    pl.col('gsc_clicks').filter(pre_may_mask).sum().alias('clicks_total'),
    pl.col('report_date').filter(pre_may_mask & (pl.col('gsc_impressions') > 0)).n_unique().alias('active_days'),
    pl.col('gsc_clicks').filter(may_mask).sum().alias('may_clicks'),
    pl.col('gsc_impressions').filter(may_mask).sum().alias('may_impressions')
])

lazy_pos_agg = lazy_daily.filter(pre_may_mask & (pl.col('gsc_avg_position') > 0)).group_by(['client_hash_id', 'content_hash_id']).agg([
    (pl.col('gsc_avg_position') * pl.col('gsc_impressions')).sum().alias('pos_num'),
    pl.col('gsc_impressions').sum().alias('pos_den')
])

agg_main = lazy_main_agg.collect()
agg_pos = lazy_pos_agg.collect()

agg_df = agg_main.join(agg_pos, on=['client_hash_id', 'content_hash_id'], how='left')
agg_df = agg_df.with_columns([
    (pl.col('pos_num') / pl.col('pos_den')).alias('weighted_position')
]).drop(['pos_num', 'pos_den'])

del agg_main, agg_pos
gc.collect()

# Derived Features & Target
agg_df = agg_df.with_columns([
    (pl.col('april_clicks') / (pl.col('feb_clicks') + 1.0)).alias('momentum'),
    ((pl.col('clicks_total') / pl.col('impressions_total')) * 100).alias('ctr'),
    (pl.col('may_clicks') < (0.8 * pl.col('april_clicks'))).cast(pl.Int64).alias('decline')
])

# Eligibility Filter & Explicit Deterministic Sorting
elig_df = agg_df.filter((pl.col('impressions_total') >= 1000) & (pl.col('april_clicks') >= 10))
elig_df = elig_df.sort(['client_hash_id', 'content_hash_id'])

elig_pd = elig_df.to_pandas()
elig_pd['weighted_position'] = elig_pd['weighted_position'].fillna(elig_pd['weighted_position'].median())

# Join dim_content parquet if available to obtain days_since_last_update
dim_content_files = list(Path(local_dir).glob('dim_content.parquet'))
if len(dim_content_files) > 0:
    try:
        dim_pl = pl.read_parquet(dim_content_files[0])
        if 'days_since_last_update' in dim_pl.columns:
            dim_df = dim_pl.select([
                pl.col('client_hash_id').cast(pl.Utf8),
                pl.col('content_hash_id').cast(pl.Utf8),
                pl.col('days_since_last_update').cast(pl.Int32)
            ]).to_pandas()
            elig_pd = elig_pd.merge(dim_df, on=['client_hash_id', 'content_hash_id'], how='left')
    except Exception:
        pass

if 'days_since_last_update' not in elig_pd.columns:
    elig_pd['days_since_last_update'] = np.nan

feature_cols = [
    'impressions_total',
    'clicks_total',
    'april_impressions',
    'april_clicks',
    'feb_clicks',
    'momentum',
    'ctr',
    'active_days',
    'weighted_position'
]

# 4. Generate Out-of-Fold (OOF) Prediction Scores using 5-Fold GroupKFold
gkf = GroupKFold(n_splits=5)
X = elig_pd[feature_cols]
y = elig_pd['decline']
groups = elig_pd['client_hash_id']

oof_scores = np.zeros(len(elig_pd))
fold_p50_scores = []
client_overlaps = []

print('\n2. Executing 5-Fold GroupKFold Cross-Validation for OOF Prediction:')
for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    tr_df = elig_pd.iloc[tr_idx]
    val_df = elig_pd.iloc[val_idx].copy()

    tr_clients = set(groups.iloc[tr_idx])
    val_clients = set(groups.iloc[val_idx])
    overlap = len(tr_clients.intersection(val_clients))
    client_overlaps.append(overlap)
    assert overlap == 0, f'Fold {fold} HAS CLIENT OVERLAP!'

    rf_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    rf_model.fit(tr_df[feature_cols], tr_df['decline'])

    val_preds = rf_model.predict_proba(val_df[feature_cols])[:, 1]
    oof_scores[val_idx] = val_preds
    val_df['rf_score'] = val_preds

    # Evaluate Precision@50 on validation fold
    val_sorted = val_df.sort_values(
        by=['rf_score', 'april_clicks', 'content_hash_id'],
        ascending=[False, False, True]
    )
    p50 = float(val_sorted.head(50)['decline'].mean())
    fold_p50_scores.append(p50)

    print(f'Fold {fold}: Train Rows={len(tr_idx):5d} | Val Rows={len(val_idx):4d} | Train Clients={len(tr_clients):2d} | Val Clients={len(val_clients):2d} | Overlap={overlap} | P@50={p50:.4f}')

elig_pd['model_score'] = oof_scores

# Quality Assertions
n_eligible = len(elig_pd)
n_clients = elig_pd['client_hash_id'].nunique()
n_oof = len(elig_pd['model_score'].dropna())
dup_predictions = elig_pd.duplicated(subset=['content_hash_id']).sum()
mean_p50 = float(np.mean(fold_p50_scores))

assert n_oof == n_eligible, f'OOF prediction count ({n_oof}) != eligible count ({n_eligible})!'
assert dup_predictions == 0, f'Duplicate page predictions detected: {dup_predictions}!'
assert max(client_overlaps) == 0, 'Client overlap detected in GroupKFold folds!'

print('\n3. OOF Generation & Validation Check:')
print(f'Eligible Pages:              {n_eligible:,}')
print(f'Distinct Clients:            {n_clients}')
print(f'OOF Predictions Count:       {n_oof:,}')
print(f'Duplicate Page Predictions:  {dup_predictions}')
print(f'Client Overlap Per Fold:     {client_overlaps}')
print(f'Min Model Score:             {elig_pd["model_score"].min():.4f}')
print(f'Max Model Score:             {elig_pd["model_score"].max():.4f}')
print(f'Reproduced Fold-Mean P@50:   {mean_p50:.4f}')

if abs(mean_p50 - 0.4440) > 0.01:
    print(f'[WARNING] Reproduced Precision@50 ({mean_p50:.4f}) differs from expected benchmark (0.4440).')
else:
    print('✅ Precision@50 successfully verified against Week 6 benchmark (~0.4440)!')

# 5. Deterministic Ranking
# Priority Order: 1. model_score desc | 2. april_clicks desc | 3. content_hash_id asc
queue_df = elig_pd.sort_values(
    by=['model_score', 'april_clicks', 'content_hash_id'],
    ascending=[False, False, True]
).reset_index(drop=True)

queue_df['rank'] = np.arange(1, len(queue_df) + 1)

# 6. Reason Codes & Action Mapping
# REASON CODE HIERARCHY (Evaluated strictly in order per page):
# 1. decline_and_stale:          model_score >= 0.50 AND days_since_last_update >= 91
# 2. decline_and_low_visibility: model_score >= 0.50 AND weighted_position > 15.0
# 3. decline_recent_update:      model_score >= 0.50 AND days_since_last_update < 91
# 4. model_signal_only:          model_score >= 0.50 (neither stale nor low-visibility)
# 5. monitor:                    model_score < 0.50
def assign_reason_code_and_action(row):
    score = row['model_score']
    stale_days = row.get('days_since_last_update', np.nan)
    pos = row['weighted_position']

    if score >= 0.50:
        if pd.notnull(stale_days) and stale_days >= 91:
            reason = 'decline_and_stale'
            action = 'refresh_review'
            priority = 'High'
        elif pos > 15.0:
            reason = 'decline_and_low_visibility'
            action = 'seo_content_review'
            priority = 'High'
        elif pd.notnull(stale_days) and stale_days < 91:
            reason = 'decline_recent_update'
            action = 'investigate_before_refresh'
            priority = 'High'
        else:
            reason = 'model_signal_only'
            action = 'manual_investigation'
            priority = 'Medium'
    else:
        reason = 'monitor'
        action = 'monitor'
        priority = 'Monitor'

    return pd.Series([reason, action, priority], index=['reason_code', 'suggested_action', 'review_priority'])

queue_df[['reason_code', 'suggested_action', 'review_priority']] = queue_df.apply(assign_reason_code_and_action, axis=1)

# Selected Output Columns (Privacy Safe: no unhashed IDs, URLs, or client names)
output_cols = [
    'rank',
    'client_hash_id',
    'content_hash_id',
    'model_score',
    'reason_code',
    'suggested_action',
    'review_priority',
    'weighted_position',
    'momentum',
    'april_clicks'
]
if 'days_since_last_update' in queue_df.columns and queue_df['days_since_last_update'].notnull().any():
    output_cols.append('days_since_last_update')

final_queue = queue_df[output_cols].copy()

# 7. Verification of Export Population & Quality Checks
export_rows = len(final_queue)
unique_content_count = final_queue['content_hash_id'].nunique()
unique_client_count = final_queue['client_hash_id'].nunique()
dup_content_count = final_queue.duplicated(subset=['content_hash_id']).sum()

print('\n4. Export Population Verification:')
print(f'Exported Row Count:              {export_rows:,} (Expected: {n_eligible:,})')
print(f'Unique content_hash_id Count:    {unique_content_count:,} (Expected: {n_eligible:,})')
print(f'Unique client_hash_id Count:     {unique_client_count} (Expected: 36)')
print(f'Duplicate content_hash_id Count: {dup_content_count} (Expected: 0)')

assert export_rows == n_eligible, f'Exported row count ({export_rows}) does not match eligible count ({n_eligible})!'
assert unique_content_count == n_eligible, f'Unique content count ({unique_content_count}) does not match eligible count ({n_eligible})!'
assert dup_content_count == 0, 'Duplicate content_hash_id detected in final queue export!'

print('\n5. Reason Code Breakdown:')
print(final_queue['reason_code'].value_counts().to_string())

print('\n6. Suggested Action Breakdown:')
print(final_queue['suggested_action'].value_counts().to_string())

print('\n7. Review Priority Breakdown:')
print(final_queue['review_priority'].value_counts().to_string())

print('\n8. Top 10 Queue Rows Preview:')
print(final_queue.head(10).to_string(index=False))

# Export Queue to work/outputs/ml_action_playbook.csv
output_path = Path('work/outputs/ml_action_playbook.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
final_queue.to_csv(output_path, index=False)
print(f'\nAction queue successfully exported to {output_path} ({len(final_queue):,} rows).')


SECTION 1: RANKED ACTIONS + REASON CODES


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Found 4 daily parquet partition files.
1. Grain check (report_date x client_hash_id x content_hash_id): duplicates = 0

2. Executing 5-Fold GroupKFold Cross-Validation for OOF Prediction:
Fold 0: Train Rows=12205 | Val Rows=4308 | Train Clients=35 | Val Clients= 1 | Overlap=0 | P@50=0.6000
Fold 1: Train Rows=13453 | Val Rows=3060 | Train Clients=34 | Val Clients= 2 | Overlap=0 | P@50=0.3200
Fold 2: Train Rows=13464 | Val Rows=3049 | Train Clients=26 | Val Clients=10 | Overlap=0 | P@50=0.5800
Fold 3: Train Rows=13465 | Val Rows=3048 | Train Clients=26 | Val Clients=10 | Overlap=0 | P@50=0.1800
Fold 4: Train Rows=13465 | Val Rows=3048 | Train Clients=23 | Val Clients=13 | Overlap=0 | P@50=0.5400

3. OOF Generation & Validation Check:
Eligible Pages:              16,513
Distinct Clients:            36
OOF Predictions Count:       16,513
Duplicate Page Predictions:  0
Client Overlap Per Fold:     [0, 0, 0, 0, 0]
Min Model Score:             0.1036
Max Model Score:             0.6599
Reprod

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended Users

The Content Action Playbook queue is designed for operational roles within digital publishing and SEO management:

- **SEO Analysts:** Who analyze search performance, monitor keyword rankings, and identify pages exhibiting organic traffic risk.
- **Content Strategists:** Who plan editorial calendars, allocate content optimization resources, and decide which portfolio assets need refresh planning.
- **Editors and Content Reviewers:** Who perform line-by-line editorial checks, update out-of-date facts, and optimize existing content for user intent.

### Intended Use: Decision-Support Prioritization

This action queue functions as a **decision-support tool** to help teams prioritize human review under constrained operational capacity.

Rather than relying on unsorted page lists or simple rule heuristics, the queue combines the **validated Week 6 Random Forest model score** (which ranks pages by estimated probability of May click decline) with **supporting review signals** (such as search visibility position and update age).

By sorting candidates deterministically, human reviewers can focus their limited time on top-ranked pages where historical performance patterns indicate elevated decline risk.

### What the Model Can Reasonably Support

Based on empirical validation across the internship dataset, the queue reasonably supports:

1. **Ranking Potentially Declining Pages:** Providing a relative risk ordering across eligible content pages.
2. **Prioritizing Operational Review:** Helping teams decide which pages to inspect *first* when reviewing large content portfolios.
3. **Surfacing Secondary Signal Triggers:** Highlighting pages that exhibit specific secondary characteristics, such as weak search visibility (`weighted_position > 15.0`).
4. **Guiding Diagnostic Investigation:** Serving as an initial entry point for human reviewers to investigate why a page might be losing search traction.

### Important Technical & Validation Limits

To ensure honest deployment, stakeholders must recognize the following empirical boundaries:

- **Validation Benchmark:** The model achieved a **Precision@50 of 0.4440** under 5-fold `GroupKFold` cross-validation by client in Week 6. This means that among the top 50 pages prioritized in validation folds, about 22 were actual declining pages (beating the simple rule baseline of 0.3920).
- **Not Production Performance:** Precision@50 = 0.4440 represents conservative out-of-sample validation performance on unseen clients, **not** guaranteed production accuracy or complete error-free classification.
- **Single Temporal Evaluation Period:** The model was trained on historical data from February 1 – April 30, 2026, and evaluated against outcomes in a single outcome window (May 1 – May 31, 2026).
- **Specific Population Scope:** Evaluated on **16,513 eligible pages across 36 clients** meeting specific activity thresholds (`impressions_total >= 1000` and `april_clicks >= 10`).
- **Probabilistic Evidence, Not Certainty:** The model score provides directional ranking evidence, not a guaranteed prediction of individual page behavior.
- **No Broader Generalization Claim:** These results reflect observed patterns within this specific portfolio and evaluation window. They should **not** be generalized to all websites, industries, or future time periods without independent out-of-sample validation.

### Staleness & Refresh Limitation (Observational Context)

In Week 4 descriptive analysis, pages stale for 91+ days exhibited an observed decline rate of **60.85%**, compared with **51.20%** for pages updated within 90 days (a +9.65 percentage point difference).

> [!WARNING]
> **Observational Association, Not Causation:** This difference is a descriptive pattern observed in historical cross-sectional data. It does **not** prove that content staleness causes traffic decline, nor does it guarantee that refreshing a stale page will restore or increase organic search traffic. Content refresh decisions must be evaluated individually by human editors.

### What the Queue Does NOT Do (Non-Goals & Guardrails)

To prevent misuse, the following boundaries are explicitly enforced:

- ❌ **Does NOT automatically edit content** — all content changes require human authorship and editorial judgment.
- ❌ **Does NOT automatically publish updates** — no automated publishing workflows are connected to this model.
- ❌ **Does NOT automatically delete or redirect pages** — pruning or URL restructuring decisions remain human choices.
- ❌ **Does NOT guarantee traffic recovery** — prioritizing a page for review does not ensure search rankings will improve.
- ❌ **Does NOT prove why a page declined** — the model ranks pages based on historical features but cannot establish root cause (e.g., algorithm updates vs. technical issues vs. competitor actions).
- ❌ **Does NOT predict Google's algorithm** — the model scores past outcome patterns within a specific client dataset, not search engine ranking algorithms.
- ❌ **Does NOT replace human review** — the queue is an input to human decision-making, never a replacement for professional domain expertise.

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

print('==================================================')
print('SECTION 2: LIGHTWEIGHT QUEUE & BOUNDARY VERIFICATION')
print('==================================================')

# 1. Load exported queue from Section 1
queue_path = Path('work/outputs/ml_action_playbook.csv')
assert queue_path.exists(), f'Queue file not found at {queue_path}! Run Section 1 code cell first.'

queue_df = pd.read_csv(queue_path)

# Enforce canonical Week 7 column names (no starter dataset columns allowed)
required_cols = ['rank', 'client_hash_id', 'content_hash_id', 'model_score', 'reason_code', 'suggested_action', 'review_priority']
for col in required_cols:
    assert col in queue_df.columns, f'Required column "{col}" missing from Section 1 export! Found columns: {queue_df.columns.tolist()}'

# 2. Population Integrity Assertions
exported_rows = len(queue_df)
unique_content = queue_df['content_hash_id'].nunique()
unique_clients = queue_df['client_hash_id'].nunique()
duplicate_content = queue_df.duplicated(subset=['content_hash_id']).sum()
null_counts = queue_df[required_cols].isnull().sum().to_dict()

print('1. Action Queue Population Integrity:')
print(f'   - Total Queue Rows:            {exported_rows:,} (Expected: 16,513)')
print(f'   - Unique content_hash_id:      {unique_content:,} (Expected: 16,513)')
print(f'   - Unique client_hash_id:       {unique_clients} (Expected: 36)')
print(f'   - Duplicate content_hash_id:   {duplicate_content} (Expected: 0)')
print(f'   - Required Field Null Counts:  {null_counts}')

assert exported_rows == 16513, f'Population mismatch: expected 16,513 rows, found {exported_rows}!'
assert unique_content == 16513, f'Unique content mismatch: expected 16,513, found {unique_content}!'
assert unique_clients == 36, f'Client count mismatch: expected 36, found {unique_clients}!'
assert duplicate_content == 0, f'Duplicate content_hash_id detected: {duplicate_content}!'
assert sum(null_counts.values()) == 0, f'Null values detected in required fields: {null_counts}'

# 3. Model Score Distribution Summary (Calculated from 16,513-row Week 7 queue)
score_stats = queue_df['model_score'].describe()
print('\n2. OOF Model Score Distribution Summary (16,513 Eligible Pages):')
print(f'   - Mean Score:   {score_stats["mean"]:.4f}')
print(f'   - Std Dev:      {score_stats["std"]:.4f}')
print(f'   - Min Score:    {score_stats["min"]:.4f}')
print(f'   - 25th Pctile:  {score_stats["25%"]:.4f}')
print(f'   - Median (50%): {score_stats["50%"]:.4f}')
print(f'   - 75th Pctile:  {score_stats["75%"]:.4f}')
print(f'   - Max Score:    {score_stats["max"]:.4f}')

# 4. Reason Code Breakdown
print('\n3. Reason Code Breakdown:')
print(queue_df['reason_code'].value_counts().to_string())

# 5. Review Priority Breakdown
priority_counts = queue_df['review_priority'].value_counts().to_dict()
print('\n4. Review Priority Breakdown:')
for priority, count in priority_counts.items():
    print(f'   - {priority:8s}: {count:6,d} pages ({count/exported_rows*100:5.2f}%)')

# 6. Data Guardrails & Leakage Safety Verification
forbidden_cols = ['may_clicks', 'may_impressions', 'decline', 'trend_direction', 'trend_pct']
present_forbidden = [c for c in forbidden_cols if c in queue_df.columns]
print('\n5. Guardrails Verification:')
print(f'   - Target / Future Columns in Queue Export: {present_forbidden}')
assert len(present_forbidden) == 0, f'Target/future leakage detected in queue export: {present_forbidden}'

print('\n✅ SECTION 2 VERIFICATION CHECKS PASSED SUCCESSFULLY!')


SECTION 2: LIGHTWEIGHT QUEUE & BOUNDARY VERIFICATION
1. Action Queue Population Integrity:
   - Total Queue Rows:            16,513 (Expected: 16,513)
   - Unique content_hash_id:      16,513 (Expected: 16,513)
   - Unique client_hash_id:       36 (Expected: 36)
   - Duplicate content_hash_id:   0 (Expected: 0)
   - Required Field Null Counts:  {'rank': 0, 'client_hash_id': 0, 'content_hash_id': 0, 'model_score': 0, 'reason_code': 0, 'suggested_action': 0, 'review_priority': 0}

2. OOF Model Score Distribution Summary (16,513 Eligible Pages):
   - Mean Score:   0.3867
   - Std Dev:      0.0803
   - Min Score:    0.1036
   - 25th Pctile:  0.3272
   - Median (50%): 0.3899
   - 75th Pctile:  0.4436
   - Max Score:    0.6599

3. Reason Code Breakdown:
reason_code
monitor                       15541
model_signal_only               723
decline_and_low_visibility      249

4. Review Priority Breakdown:
   - Monitor : 15,541 pages (94.11%)
   - Medium  :    723 pages ( 4.38%)
   - High    :   

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.